# 02 - Cohort and Exposure Construction

Short summary:
[02 - Cohort and Exposure Construction (SHORT)](../docs/02_cohort_and_exposures_SHORT.md)

---

## 1. Objective
Construct the COPD ICU cohort and derive pre-ICU RAAS inhibitor exposure indicators for downstream cohort assembly and analysis.

---

## 2. Data Processing Steps

Prescription flags represent same-admission records with `starttime <= intime`, not verified administration or concurrent treatment. Legacy monotherapy/dual-exposure labels identify the presence of qualifying records for one or both classes only. Each subclass analysis uses neither qualifying order as reference; no direct ACEi-versus-ARB contrast is estimated.


### 2.1 Common BigQuery Utilities


In [1]:
# Shared ADC client with US location, dry runs and per-job byte guards.
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from notebook_bq_runtime import client_for_notebook, run_sql_script, query_to_df
client = client_for_notebook()
print("Connected to BigQuery project: mimic-iv-portfolio (US)")

Connected to BigQuery project: mimic-iv-portfolio (US)


### 2.2 Build Cohort and Exposure Tables in BigQuery
#### 2.2.1 Build COPD ICU Cohort


In [2]:
import pandas as pd

# Build the COPD ICU cohort in BigQuery using ICD-9/10 diagnosis codes
# Output: copd_raas.cohort_copd
run_sql_script("../sql/02_build_cohort_copd.sql")

Executed SQL script: 02_build_cohort_copd.sql


#### 2.2.2 Define Binary Pre-ICU RAAS Inhibitor Exposure


In [3]:
# Create binary pre-ICU RAAS inhibitor exposure flag (raas_pre_icu)
# Output: copd_raas.cohort_copd_raas
run_sql_script("../sql/02_exposure_raas.sql")

Executed SQL script: 02_exposure_raas.sql


#### 2.2.3 Create Detailed RAAS Inhibitor Exposure Flags


This step defines medication-based pre-ICU RAAS exposure variables for the COPD ICU cohort.

- **SQL script:** [02_exposure_raas_detailed.sql](../sql/02_exposure_raas_detailed.sql)
- **Output table:** `copd_raas.cohort_copd_raas_detailed`

The SQL script identifies ACE inhibitor (ACEi) and angiotensin receptor blocker (ARB)
prescriptions from the MIMIC-IV `mimiciv_3_1_hosp.prescriptions` table using drug-name
pattern matching, and links them to ICU stays via patient and admission identifiers.

Only same-admission prescription records starting on or before ICU admission qualify.
This defines the order-record window; it does not verify administration or rule out
reverse causation or confounding.

At the ICU-stay level, the script derives indicators for qualifying ACEi and ARB
prescription orders, any-class and both-class flags, and mutually exclusive order groups
(ACEi only, ARB only, both, or neither). Both-class records do not demonstrate simultaneous
treatment; neither means no qualifying record under this definition.

In [4]:
# Execute SQL script to create detailed pre-ICU RAAS inhibitor exposure flags (ACEi / ARB / both / neither / monotherapy)
# Output: copd_raas.cohort_copd_raas_detailed
run_sql_script("../sql/02_exposure_raas_detailed.sql")

Executed SQL script: 02_exposure_raas_detailed.sql


### 2.3 Sanity Checks
#### 2.3.1 Binary Pre-ICU RAAS Inhibitor Exposure


A simple check is performed to confirm the distribution of the binary pre-ICU RAAS inhibitor exposure flag (raas_pre_icu) derived from medication records.

Expected (and observed) distribution:
- raas_pre_icu = 0: 11,018
- raas_pre_icu = 1: 946

In [5]:
# Simple pre-ICU RAAS inhibitor exposure Flag Sanity Check
# From: copd_raas.cohort_copd_raas (02_exposure_raas.sql)

sql_simple = """
SELECT 
  raas_pre_icu,
  COUNT(*) AS n
FROM `mimic-iv-portfolio.copd_raas.cohort_copd_raas`
GROUP BY raas_pre_icu
ORDER BY raas_pre_icu
"""

simple_counts = query_to_df(sql_simple)
df = simple_counts.copy()

# Replace 0/1 directly in the column
df["exposure_group"] = df["raas_pre_icu"].map({
    0: "No RAAS inhibitor use before or at ICU admission",
    1: "RAAS inhibitor use before or at ICU admission"
})

# Stable aggregate output without patient-level identifiers.
df[["exposure_group", "n"]]


,exposure_group,n
0,No RAAS inhibitor use before or at ICU admission,11018
1,RAAS inhibitor use before or at ICU admission,946


#### 2.3.2 Cohort Integrity Check: One Row per ICU Stay


In [6]:
# Sanity check: one row per ICU stay
query_to_df("""
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT stay_id) AS n_unique_stays
FROM `mimic-iv-portfolio.copd_raas.cohort_copd`
""")



,n_rows,n_unique_stays
0,11964,11964


#### 2.3.3 Detailed Exposure Group Counts


In [7]:
# Quick sanity check: number of patients per exposure group
sql_check = """
SELECT exposure_group_4cat, COUNT(*) AS n
FROM `mimic-iv-portfolio.copd_raas.cohort_copd_raas_detailed`
GROUP BY exposure_group_4cat
ORDER BY exposure_group_4cat
"""
exposure_counts = query_to_df(sql_check)
exposure_counts

,exposure_group_4cat,n
0,acei_only,643
1,arb_only,290
2,both,13
3,neither,11018


---

## 3. Output Dataset

**Input Dataset**
- `mimic-iv-portfolio.copd_raas.cohort_icu`
- MIMIC-IV diagnosis and prescription tables in BigQuery.

**Output Dataset**
- `mimic-iv-portfolio.copd_raas.cohort_copd`
- `mimic-iv-portfolio.copd_raas.cohort_copd_raas`
- `mimic-iv-portfolio.copd_raas.cohort_copd_raas_detailed`

**Downstream Usage**
- These tables define the COPD ICU cohort and pre-ICU RAAS exposure indicators used by baseline cohort construction, subgroup analyses, and validation workflows.


---

## Next Step

Proceed to:
- [03a - Baseline Cohort Construction](03a_baseline.ipynb)

This next notebook assembles baseline demographics, eligibility fields, and exposure inputs for analysis.
